# 🚀 TalentAI: Recruitment Intelligence Agent
## Built by **Aigenthix** | Powered by LangGraph + Groq + Gradio

---

### System Architecture
- **LLM**: Groq → llama-3.1-8b-instant
- **Orchestration**: LangGraph Multi-Node Workflow
- **UI**: Gradio (Enterprise HR Dashboard)
- **Resume Parsing**: PyMuPDF + python-docx
- **Intelligence**: Candidate Scoring, Matching, Scheduling, HR Comms

---

**Run each cell in order. Set your Groq API key in Cell 2.**

In [ ]:
# ============================================================
# CELL 1: INSTALL ALL DEPENDENCIES
# ============================================================
print("📦 Installing TalentAI dependencies...")

!pip install -q groq langchain langchain-groq langgraph gradio
!pip install -q pymupdf python-docx pypdf2 pdfplumber
!pip install -q pandas numpy scikit-learn matplotlib plotly
!pip install -q python-dateutil

print("✅ All dependencies installed successfully!")

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION & API KEY SETUP
# ============================================================
import os
from google.colab import userdata

# ─── Option A: Use Colab Secrets (Recommended) ───────────────
# Add your key as 'GROQ_API_KEY' in Colab Secrets panel (🔑 icon)
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Groq API key loaded from Colab Secrets")
except:
    # ─── Option B: Paste directly ────────────────────────────
    os.environ["GROQ_API_KEY"] = "gsk_YOUR_GROQ_API_KEY_HERE"  # <── PASTE HERE
    print("⚠️  Using hardcoded API key — consider using Colab Secrets for security")

GROQ_MODEL = "llama-3.1-8b-instant"
print(f"🤖 Model: {GROQ_MODEL}")

In [ ]:
# ============================================================
# CELL 3: CORE IMPORTS
# ============================================================
import json
import re
import io
import time
import random
import hashlib
from typing import TypedDict, List, Dict, Any, Optional, Annotated
from datetime import datetime, timedelta
from pathlib import Path
import operator

import pandas as pd
import numpy as np

# LangGraph
from langgraph.graph import StateGraph, END

# Groq
from groq import Groq

# Resume parsing
import fitz  # PyMuPDF
import docx
import pdfplumber

# Gradio
import gradio as gr

print("✅ All imports successful!")

In [ ]:
# ============================================================
# CELL 4: GROQ CLIENT + LLM UTILITIES
# ============================================================

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def groq_call(system_prompt: str, user_prompt: str, temperature: float = 0.3, max_tokens: int = 2048) -> str:
    """Core Groq LLM call with error handling and retry logic."""
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt == 2:
                return f"Error: {str(e)}"
            time.sleep(2 ** attempt)

def groq_json_call(system_prompt: str, user_prompt: str) -> dict:
    """Groq call that enforces JSON output."""
    system_prompt_json = system_prompt + "\n\nIMPORTANT: Respond ONLY with valid JSON. No markdown, no backticks, no explanation."
    raw = groq_call(system_prompt_json, user_prompt)
    # Clean and parse
    raw = re.sub(r'^```[a-z]*\n?', '', raw.strip())
    raw = re.sub(r'\n?```$', '', raw.strip())
    try:
        return json.loads(raw)
    except:
        # Try to extract JSON
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
        return {"raw": raw, "parse_error": True}

print("✅ Groq client initialized!")

In [ ]:
# ============================================================
# CELL 5: RESUME PARSING ENGINE
# ============================================================

def extract_text_from_pdf(file_path: str) -> str:
    """Extract text from PDF using pdfplumber with PyMuPDF fallback."""
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception:
        pass

    if not text.strip():
        try:
            doc = fitz.open(file_path)
            for page in doc:
                text += page.get_text() + "\n"
            doc.close()
        except Exception as e:
            text = f"PDF extraction error: {e}"
    return text.strip()

def extract_text_from_docx(file_path: str) -> str:
    """Extract text from DOCX file."""
    try:
        document = docx.Document(file_path)
        paragraphs = [p.text for p in document.paragraphs if p.text.strip()]
        # Also extract from tables
        for table in document.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        paragraphs.append(cell.text.strip())
        return "\n".join(paragraphs)
    except Exception as e:
        return f"DOCX extraction error: {e}"

def extract_resume_text(file_path: str) -> str:
    """Unified resume text extractor."""
    ext = Path(file_path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(file_path)
    elif ext in [".docx", ".doc"]:
        return extract_text_from_docx(file_path)
    elif ext == ".txt":
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    else:
        return "Unsupported file format"

def parse_candidate_features(resume_text: str, candidate_name: str = "Unknown") -> dict:
    """Use Groq LLM to extract structured candidate features from resume text."""
    system_prompt = """You are an expert HR AI that extracts structured candidate data from resumes.
Extract all available information and return comprehensive JSON."""

    user_prompt = f"""Extract structured data from this resume:

{resume_text[:4000]}

Return JSON with these fields:
{{
  "name": "candidate full name",
  "email": "email address or null",
  "phone": "phone number or null",
  "location": "city/country or null",
  "summary": "2-3 sentence professional summary",
  "total_experience_years": number,
  "current_role": "current/most recent job title",
  "current_company": "current/most recent company",
  "technical_skills": ["skill1", "skill2", ...],
  "soft_skills": ["skill1", "skill2", ...],
  "tools_technologies": ["tool1", "tool2", ...],
  "programming_languages": ["lang1", "lang2", ...],
  "education": [
    {{"degree": "degree name", "institution": "name", "year": "year", "field": "field of study"}}
  ],
  "work_history": [
    {{"title": "role", "company": "company", "duration": "period", "responsibilities": ["resp1"]}}
  ],
  "certifications": ["cert1", "cert2"],
  "languages": ["language1"],
  "achievements": ["achievement1"],
  "industry_domain": "primary industry domain",
  "seniority_level": "intern/junior/mid/senior/lead/manager/director"
}}"""

    result = groq_json_call(system_prompt, user_prompt)
    if "parse_error" in result:
        result = {"name": candidate_name, "summary": resume_text[:500], "technical_skills": [], "total_experience_years": 0}
    result["raw_text"] = resume_text[:3000]
    result["candidate_id"] = hashlib.md5(resume_text[:200].encode()).hexdigest()[:8].upper()
    return result

print("✅ Resume parsing engine ready!")

In [ ]:
# ============================================================
# CELL 6: JD UNDERSTANDING ENGINE
# ============================================================

def parse_job_description(jd_text: str) -> dict:
    """Use Groq to extract structured requirements from job description."""
    system_prompt = """You are a senior talent acquisition expert. Extract comprehensive structured requirements from job descriptions."""

    user_prompt = f"""Analyze this job description and extract structured requirements:

{jd_text[:3000]}

Return JSON:
{{
  "job_title": "exact job title",
  "company": "company name if mentioned",
  "department": "department",
  "location": "location/remote",
  "employment_type": "full-time/part-time/contract",
  "seniority_level": "junior/mid/senior/lead/manager",
  "required_experience_years": number,
  "max_experience_years": number or null,
  "required_technical_skills": ["skill1", "skill2"],
  "preferred_technical_skills": ["skill1"],
  "required_soft_skills": ["skill1"],
  "required_tools": ["tool1"],
  "required_programming_languages": ["lang1"],
  "required_education": "degree requirement",
  "required_certifications": ["cert1"],
  "key_responsibilities": ["resp1", "resp2"],
  "industry_domain": "domain",
  "salary_range": "range if mentioned",
  "must_have_keywords": ["keyword1"],
  "nice_to_have_keywords": ["keyword1"],
  "role_summary": "2 sentence role summary"
}}"""

    result = groq_json_call(system_prompt, user_prompt)
    if "parse_error" in result:
        result = {"job_title": "Unknown Role", "required_technical_skills": [], "required_experience_years": 0}
    result["raw_jd"] = jd_text
    return result

print("✅ JD understanding engine ready!")

In [ ]:
# ============================================================
# CELL 7: CANDIDATE SCORING & MATCHING ENGINE
# ============================================================

def compute_skill_overlap(candidate_skills: list, required_skills: list) -> float:
    """Compute normalized skill overlap score."""
    if not required_skills:
        return 0.5
    candidate_lower = {s.lower().strip() for s in candidate_skills if isinstance(s, str)}
    matched = 0
    for skill in required_skills:
        skill_l = skill.lower().strip()
        # Exact or partial match
        if any(skill_l in c or c in skill_l for c in candidate_lower):
            matched += 1
    return min(matched / len(required_skills), 1.0)

def compute_experience_score(candidate_years: float, required_years: float, max_years: float = None) -> float:
    """Score candidate experience vs requirements."""
    if required_years == 0:
        return 0.8
    if candidate_years >= required_years:
        if max_years and candidate_years > max_years * 1.5:
            return 0.75  # Overqualified
        return min(1.0, 0.7 + (candidate_years - required_years) * 0.05)
    else:
        ratio = candidate_years / required_years
        return max(0.1, ratio * 0.8)

def score_candidate(candidate: dict, jd: dict) -> dict:
    """Multi-dimensional candidate scoring against JD."""
    # Collect all candidate skills
    all_candidate_skills = (
        candidate.get("technical_skills", []) +
        candidate.get("tools_technologies", []) +
        candidate.get("programming_languages", [])
    )

    # 1. Technical Skill Match (35%)
    required_skills = jd.get("required_technical_skills", []) + jd.get("required_programming_languages", [])
    tech_score = compute_skill_overlap(all_candidate_skills, required_skills)

    # 2. Preferred Skill Match (15%)
    pref_score = compute_skill_overlap(all_candidate_skills, jd.get("preferred_technical_skills", []))

    # 3. Experience Match (25%)
    exp_score = compute_experience_score(
        float(candidate.get("total_experience_years", 0)),
        float(jd.get("required_experience_years", 0)),
        jd.get("max_experience_years")
    )

    # 4. Tool Match (10%)
    tool_score = compute_skill_overlap(all_candidate_skills, jd.get("required_tools", []))

    # 5. Seniority Alignment (10%)
    seniority_map = {"intern": 0, "junior": 1, "mid": 2, "senior": 3, "lead": 4, "manager": 5, "director": 6}
    cand_level = seniority_map.get(candidate.get("seniority_level", "mid").lower(), 2)
    req_level = seniority_map.get(jd.get("seniority_level", "mid").lower(), 2)
    level_diff = abs(cand_level - req_level)
    seniority_score = max(0.2, 1.0 - level_diff * 0.25)

    # 5. Domain alignment (5%)
    domain_score = 0.8 if candidate.get("industry_domain", "").lower() == jd.get("industry_domain", "").lower() else 0.4

    # Weighted total
    total = (
        tech_score * 0.35 +
        pref_score * 0.15 +
        exp_score * 0.25 +
        tool_score * 0.10 +
        seniority_score * 0.10 +
        domain_score * 0.05
    )

    # Skill gap analysis
    all_candidate_lower = {s.lower() for s in all_candidate_skills}
    skill_gaps = [
        s for s in required_skills
        if not any(s.lower() in c or c in s.lower() for c in all_candidate_lower)
    ]

    # Rating label
    pct = round(total * 100)
    if pct >= 80: rating = "⭐ Excellent Match"
    elif pct >= 65: rating = "✅ Good Match"
    elif pct >= 50: rating = "⚠️ Partial Match"
    else: rating = "❌ Weak Match"

    return {
        "candidate_id": candidate.get("candidate_id", "N/A"),
        "name": candidate.get("name", "Unknown"),
        "email": candidate.get("email", "N/A"),
        "current_role": candidate.get("current_role", "N/A"),
        "experience_years": candidate.get("total_experience_years", 0),
        "seniority_level": candidate.get("seniority_level", "N/A"),
        "overall_match_pct": pct,
        "rating": rating,
        "tech_score": round(tech_score * 100),
        "experience_score": round(exp_score * 100),
        "tool_score": round(tool_score * 100),
        "seniority_score": round(seniority_score * 100),
        "skill_gaps": skill_gaps[:8],
        "matched_skills": [
            s for s in required_skills
            if any(s.lower() in c or c in s.lower() for c in all_candidate_lower)
        ][:10],
        "technical_skills": candidate.get("technical_skills", [])[:10],
        "summary": candidate.get("summary", ""),
        "location": candidate.get("location", "N/A"),
    }

def shortlist_candidates(scored_candidates: list, n: int = 5, threshold: int = 40) -> list:
    """Shortlist top N candidates above threshold."""
    eligible = [c for c in scored_candidates if c["overall_match_pct"] >= threshold]
    eligible.sort(key=lambda x: x["overall_match_pct"], reverse=True)
    return eligible[:n]

print("✅ Scoring and matching engine ready!")

In [ ]:
# ============================================================
# CELL 8: INTERVIEW QUESTION GENERATION
# ============================================================

def generate_interview_questions(candidate: dict, jd: dict, interview_type: str = "technical") -> str:
    """Generate tailored interview questions using Groq."""
    system_prompt = """You are a senior interviewer and talent assessment expert at a top-tier company.
Generate targeted, insightful interview questions that assess both technical competence and cultural fit."""

    candidate_summary = f"""
Candidate: {candidate.get('name')}
Role: {candidate.get('current_role')}
Experience: {candidate.get('experience_years')} years
Skills: {', '.join(candidate.get('technical_skills', [])[:8])}
Skill Gaps: {', '.join(candidate.get('skill_gaps', [])[:5])}
Match Score: {candidate.get('overall_match_pct')}%
"""

    jd_summary = f"""
Role: {jd.get('job_title')}
Required Skills: {', '.join(jd.get('required_technical_skills', [])[:8])}
Experience Required: {jd.get('required_experience_years')} years
Key Responsibilities: {', '.join(jd.get('key_responsibilities', [])[:5])}
"""

    type_instructions = {
        "technical": "Focus on technical depth, system design, coding, problem-solving. Include 2 technical scenarios.",
        "hr": "Focus on cultural fit, motivation, values, career goals, and behavioral patterns.",
        "managerial": "Focus on leadership, team management, conflict resolution, strategic thinking.",
        "mixed": "Balanced mix of technical, behavioral, and situational questions."
    }

    user_prompt = f"""Generate a comprehensive interview guide for this candidate:

CANDIDATE PROFILE:
{candidate_summary}

JOB REQUIREMENTS:
{jd_summary}

INTERVIEW TYPE: {interview_type.upper()}
FOCUS: {type_instructions.get(interview_type, type_instructions['mixed'])}

Generate:
1. **Opening Questions** (2 warm-up questions)
2. **Core Competency Questions** (4 role-specific questions)
3. **Behavioral Questions** (3 STAR-format questions)
4. **Skill Gap Probe Questions** (2 questions targeting identified gaps)
5. **Scenario/Case Questions** (2 practical scenarios)
6. **Closing Questions** (2 questions)

For each question, add an [Evaluation Criteria] note.

End with a **Evaluation Scorecard Template** with 5 key dimensions rated 1-5."""

    return groq_call(system_prompt, user_prompt, temperature=0.4, max_tokens=2500)

print("✅ Interview question generator ready!")

In [ ]:
# ============================================================
# CELL 9: INTERVIEW SCHEDULING ENGINE
# ============================================================

def generate_interview_schedule(shortlisted: list, jd: dict, panel_size: int = 2) -> str:
    """Generate a structured interview schedule for shortlisted candidates."""
    system_prompt = """You are an expert HR coordinator specializing in interview logistics and scheduling optimization."""

    candidates_info = "\n".join([
        f"- {c.get('name')} (Match: {c.get('overall_match_pct')}%, Role: {c.get('current_role')})"
        for c in shortlisted
    ])

    base_date = datetime.now() + timedelta(days=3)

    user_prompt = f"""Create a detailed interview schedule for these shortlisted candidates:

CANDIDATES ({len(shortlisted)} total):
{candidates_info}

ROLE: {jd.get('job_title')}
PANEL SIZE: {panel_size} interviewers per session
STARTING DATE: {base_date.strftime('%A, %B %d, %Y')}

Create a schedule with:
1. **Interview Calendar** — Day-by-day time slots for each candidate
   - Round 1: HR Screening (30 min)
   - Round 2: Technical (60 min)
   - Round 3: Managerial (45 min)
   - Round 4: Final (HR + Leadership, 45 min) — for top 2 candidates only

2. **Panel Composition** — Suggested interviewer roles per round

3. **Logistics Checklist** — What to prepare before each interview

4. **Decision Timeline** — When to expect hiring decision

Format as a professional, calendar-ready schedule. Use clear tables and sections."""

    return groq_call(system_prompt, user_prompt, temperature=0.2, max_tokens=2000)

print("✅ Scheduling engine ready!")

In [ ]:
# ============================================================
# CELL 10: HR COMMUNICATION GENERATOR
# ============================================================

def generate_hr_communication(comm_type: str, candidate: dict, jd: dict, tone: str = "professional") -> str:
    """Generate HR communications for various scenarios."""
    system_prompt = f"""You are a senior HR Business Partner with excellent written communication skills.
You write clear, warm, and professional HR communications.
Tone: {tone}. Always personalize communications."""

    comm_templates = {
        "shortlist_invite": "Interview invitation for shortlisted candidate",
        "rejection": "Professional, empathetic rejection email",
        "offer_letter": "Formal job offer letter",
        "onboarding_welcome": "Warm onboarding welcome message",
        "follow_up": "Post-interview follow-up email",
        "assessment_task": "Technical assessment assignment email",
        "reference_check": "Reference check request email"
    }

    user_prompt = f"""Generate a {comm_templates.get(comm_type, comm_type)} for:

Candidate: {candidate.get('name', 'Candidate')}
Email: {candidate.get('email', '[email]')}
Role Applied: {jd.get('job_title', 'the position')}
Company: {jd.get('company', 'our company')}
Match Score: {candidate.get('overall_match_pct', 'N/A')}%
Current Role: {candidate.get('current_role', 'N/A')}

Communication Type: {comm_type.replace('_', ' ').title()}
Tone: {tone}

Include:
- Personalized subject line
- Proper salutation
- Core message (specific to type)
- Clear call to action
- Professional sign-off
- [Placeholder] for details to fill in

Make it warm, human, and authentic. Avoid generic corporate language."""

    return groq_call(system_prompt, user_prompt, temperature=0.5, max_tokens=1500)

def generate_job_description_from_scratch(role_title: str, industry: str, experience_level: str, key_skills: str) -> str:
    """AI-powered JD generator."""
    system_prompt = """You are a talent acquisition expert who writes compelling, inclusive, and accurate job descriptions that attract top talent."""

    user_prompt = f"""Write a complete job description for:

Role: {role_title}
Industry: {industry}
Level: {experience_level}
Key Skills Required: {key_skills}

Include: Company overview placeholder, Role Summary, Key Responsibilities (8-10 bullets), Required Qualifications, Preferred Qualifications, What We Offer, Equal Opportunity Statement.

Write in an engaging, modern style. Avoid exclusionary language."""

    return groq_call(system_prompt, user_prompt, temperature=0.4, max_tokens=2000)

print("✅ HR Communication generator ready!")

In [ ]:
# ============================================================
# CELL 11: HIRING ANALYTICS ENGINE
# ============================================================

def generate_hiring_analytics(scored_candidates: list, jd: dict) -> str:
    """Generate comprehensive hiring analytics and insights."""
    if not scored_candidates:
        return "No candidates available for analytics."

    # Compute pipeline stats
    scores = [c["overall_match_pct"] for c in scored_candidates]
    avg_score = np.mean(scores)
    excellent = sum(1 for s in scores if s >= 80)
    good = sum(1 for s in scores if 65 <= s < 80)
    partial = sum(1 for s in scores if 50 <= s < 65)
    weak = sum(1 for s in scores if s < 50)

    # Skill frequency
    all_skills = []
    for c in scored_candidates:
        all_skills.extend(c.get("technical_skills", []))
    skill_freq = pd.Series(all_skills).value_counts().head(10).to_dict() if all_skills else {}

    # Common skill gaps
    all_gaps = []
    for c in scored_candidates:
        all_gaps.extend(c.get("skill_gaps", []))
    gap_freq = pd.Series(all_gaps).value_counts().head(8).to_dict() if all_gaps else {}

    pipeline_data = f"""
PIPELINE SUMMARY:
- Total Candidates: {len(scored_candidates)}
- Average Match Score: {avg_score:.1f}%
- Excellent Match (80%+): {excellent} candidates
- Good Match (65-79%): {good} candidates
- Partial Match (50-64%): {partial} candidates
- Weak Match (<50%): {weak} candidates

TOP CANDIDATE SKILLS IN POOL: {list(skill_freq.keys())[:8]}
COMMON SKILL GAPS ACROSS POOL: {list(gap_freq.keys())[:6]}

CANDIDATES BY EXPERIENCE:
{chr(10).join([f'  - {c["name"]}: {c["experience_years"]}y | {c["overall_match_pct"]}% match | {c["seniority_level"]}' for c in sorted(scored_candidates, key=lambda x: x['overall_match_pct'], reverse=True)[:8]])}
"""

    system_prompt = """You are a Chief People Officer and Talent Analytics expert providing data-driven hiring insights."""

    user_prompt = f"""Analyze this hiring pipeline and provide executive-level insights:

ROLE: {jd.get('job_title')}
{pipeline_data}

Provide:
1. **Executive Summary** (3 bullet points)
2. **Pipeline Health Assessment** (quality, diversity, readiness)
3. **Top Candidate Highlights** (spotlight top 3)
4. **Skill Market Analysis** (supply vs demand for required skills)
5. **Hiring Risk Assessment** (potential challenges)
6. **Strategic Recommendations** (sourcing, timeline, compensation)
7. **Next Steps** (prioritized action list)

Be specific, data-driven, and actionable."""

    analytics_text = groq_call(system_prompt, user_prompt, temperature=0.3, max_tokens=2000)
    return f"""📊 HIRING ANALYTICS DASHBOARD\n{'='*60}\n\n{pipeline_data}\n\n🤖 AI-POWERED INSIGHTS:\n{analytics_text}"""

def generate_workforce_plan(role: str, industry: str, team_size: int, growth_target: str) -> str:
    """Generate strategic workforce planning insights."""
    system_prompt = """You are a Strategic Workforce Planning expert and organizational design consultant."""

    user_prompt = f"""Develop a strategic workforce plan for:

Role/Function: {role}
Industry: {industry}
Current Team Size: {team_size}
Growth Target: {growth_target}

Provide:
1. **Talent Demand Forecast** (6-12 months)
2. **Build vs Buy vs Borrow Analysis** (internal vs external hiring vs contractors)
3. **Critical Skill Needs** (emerging skills to prioritize)
4. **Sourcing Channel Strategy** (job boards, LinkedIn, referrals, campus, etc.)
5. **Hiring Timeline** (phased approach with milestones)
6. **Budget Estimation** (relative cost categories)
7. **Risk Mitigation** (retention, skill obsolescence, market competition)
8. **KPIs to Track** (time-to-hire, quality of hire, retention rate)

Be strategic and specific to the industry."""

    return groq_call(system_prompt, user_prompt, temperature=0.4, max_tokens=2000)

print("✅ Analytics engine ready!")

In [ ]:
# ============================================================
# CELL 12: LANGGRAPH STATE & WORKFLOW
# ============================================================

class RecruitmentState(TypedDict):
    """Master state for the LangGraph recruitment workflow."""
    # Inputs
    resume_texts: List[str]
    resume_filenames: List[str]
    job_description: str
    job_title: str
    interview_type: str
    shortlist_count: int
    comm_type: str
    comm_tone: str

    # Parsed data
    parsed_candidates: List[dict]
    parsed_jd: dict

    # Results
    scored_candidates: List[dict]
    shortlisted_candidates: List[dict]

    # Outputs
    interview_questions: str
    interview_schedule: str
    hr_communication: str
    analytics_report: str

    # Meta
    status: str
    errors: List[str]


# ─── LangGraph Nodes ──────────────────────────────────────────

def node_parse_resumes(state: RecruitmentState) -> RecruitmentState:
    """Node 1: Parse and extract features from all resumes."""
    parsed = []
    for i, (text, fname) in enumerate(zip(state["resume_texts"], state["resume_filenames"])):
        if text.strip():
            features = parse_candidate_features(text, fname)
            parsed.append(features)
    state["parsed_candidates"] = parsed
    state["status"] = f"Parsed {len(parsed)} resumes"
    return state

def node_parse_jd(state: RecruitmentState) -> RecruitmentState:
    """Node 2: Parse and structure the job description."""
    state["parsed_jd"] = parse_job_description(state["job_description"])
    state["status"] = f"JD parsed: {state['parsed_jd'].get('job_title', 'Unknown')}"
    return state

def node_score_candidates(state: RecruitmentState) -> RecruitmentState:
    """Node 3: Score all candidates against JD."""
    scored = []
    for candidate in state["parsed_candidates"]:
        score = score_candidate(candidate, state["parsed_jd"])
        scored.append(score)
    # Sort by score descending
    scored.sort(key=lambda x: x["overall_match_pct"], reverse=True)
    state["scored_candidates"] = scored
    state["status"] = f"Scored {len(scored)} candidates"
    return state

def node_shortlist(state: RecruitmentState) -> RecruitmentState:
    """Node 4: Shortlist top candidates."""
    n = state.get("shortlist_count", 5)
    state["shortlisted_candidates"] = shortlist_candidates(state["scored_candidates"], n=n)
    state["status"] = f"Shortlisted {len(state['shortlisted_candidates'])} candidates"
    return state

def node_generate_questions(state: RecruitmentState) -> RecruitmentState:
    """Node 5: Generate interview questions for top candidate."""
    if state["shortlisted_candidates"]:
        top = state["shortlisted_candidates"][0]
        state["interview_questions"] = generate_interview_questions(
            top, state["parsed_jd"], state.get("interview_type", "mixed")
        )
    return state

def node_schedule_interviews(state: RecruitmentState) -> RecruitmentState:
    """Node 6: Generate interview schedule."""
    if state["shortlisted_candidates"]:
        state["interview_schedule"] = generate_interview_schedule(
            state["shortlisted_candidates"], state["parsed_jd"]
        )
    return state

def node_generate_comms(state: RecruitmentState) -> RecruitmentState:
    """Node 7: Generate HR communications."""
    if state["shortlisted_candidates"]:
        top = state["shortlisted_candidates"][0]
        state["hr_communication"] = generate_hr_communication(
            state.get("comm_type", "shortlist_invite"),
            top, state["parsed_jd"],
            state.get("comm_tone", "professional")
        )
    return state

def node_analytics(state: RecruitmentState) -> RecruitmentState:
    """Node 8: Generate hiring analytics."""
    state["analytics_report"] = generate_hiring_analytics(
        state["scored_candidates"], state["parsed_jd"]
    )
    return state


# ─── Build LangGraph ─────────────────────────────────────────

def build_recruitment_graph():
    """Build the full LangGraph recruitment workflow."""
    graph = StateGraph(RecruitmentState)

    graph.add_node("parse_resumes", node_parse_resumes)
    graph.add_node("parse_jd", node_parse_jd)
    graph.add_node("score_candidates", node_score_candidates)
    graph.add_node("shortlist", node_shortlist)
    graph.add_node("generate_questions", node_generate_questions)
    graph.add_node("schedule_interviews", node_schedule_interviews)
    graph.add_node("generate_comms", node_generate_comms)
    graph.add_node("analytics", node_analytics)

    graph.set_entry_point("parse_resumes")
    graph.add_edge("parse_resumes", "parse_jd")
    graph.add_edge("parse_jd", "score_candidates")
    graph.add_edge("score_candidates", "shortlist")
    graph.add_edge("shortlist", "generate_questions")
    graph.add_edge("generate_questions", "schedule_interviews")
    graph.add_edge("schedule_interviews", "generate_comms")
    graph.add_edge("generate_comms", "analytics")
    graph.add_edge("analytics", END)

    return graph.compile()

recruitment_graph = build_recruitment_graph()
print("✅ LangGraph workflow compiled successfully!")

In [ ]:
# ============================================================
# CELL 13: GRADIO UI HELPERS & FORMATTERS
# ============================================================

def format_scorecard_html(scored_candidates: list) -> str:
    """Generate HTML scorecard table for Gradio."""
    if not scored_candidates:
        return "<p style='color:#888;text-align:center;padding:40px'>No candidates scored yet. Upload resumes and run analysis.</p>"

    rows = ""
    for i, c in enumerate(scored_candidates, 1):
        score = c['overall_match_pct']
        color = '#22c55e' if score >= 80 else '#f59e0b' if score >= 65 else '#ef4444' if score >= 50 else '#94a3b8'
        bar_width = score
        skills_html = " ".join([f"<span style='background:#e0f2fe;color:#0369a1;border-radius:4px;padding:2px 7px;font-size:11px;margin:2px;display:inline-block'>{s}</span>" for s in c.get('technical_skills', [])[:5]])
        gaps_html = " ".join([f"<span style='background:#fef9c3;color:#854d0e;border-radius:4px;padding:2px 7px;font-size:11px;margin:2px;display:inline-block'>{s}</span>" for s in c.get('skill_gaps', [])[:4]])

        rows += f"""
        <tr style='border-bottom:1px solid #f1f5f9;hover-background:#f8fafc'>
          <td style='padding:14px 10px;font-weight:700;color:#334155'>#{i}</td>
          <td style='padding:14px 10px'>
            <div style='font-weight:600;color:#1e293b;font-size:14px'>{c.get('name','N/A')}</div>
            <div style='color:#64748b;font-size:12px'>{c.get('current_role','N/A')}</div>
            <div style='color:#94a3b8;font-size:11px'>📍 {c.get('location','N/A')} | ID: {c.get('candidate_id','')}</div>
          </td>
          <td style='padding:14px 10px'>
            <div style='font-size:24px;font-weight:800;color:{color}'>{score}%</div>
            <div style='background:#f1f5f9;border-radius:4px;height:6px;width:100px;margin-top:4px'>
              <div style='background:{color};border-radius:4px;height:6px;width:{bar_width}px'></div>
            </div>
            <div style='font-size:11px;color:{color};margin-top:3px'>{c.get('rating','')}</div>
          </td>
          <td style='padding:14px 10px;color:#475569;font-size:13px'>{c.get('experience_years',0)} yrs | {c.get('seniority_level','N/A')}</td>
          <td style='padding:14px 10px'>
            <div style='font-size:11px;color:#64748b;margin-bottom:3px'>Skills</div>{skills_html}
          </td>
          <td style='padding:14px 10px'>
            <div style='font-size:11px;color:#64748b;margin-bottom:3px'>Gaps</div>{gaps_html if gaps_html else '<span style="color:#22c55e;font-size:12px">✓ No critical gaps</span>'}
          </td>
        </tr>"""

    return f"""
    <div style='font-family:"DM Sans",sans-serif;overflow-x:auto'>
    <table style='width:100%;border-collapse:collapse;font-size:13px'>
      <thead>
        <tr style='background:#f8fafc;border-bottom:2px solid #e2e8f0'>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>#</th>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>CANDIDATE</th>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>MATCH SCORE</th>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>EXPERIENCE</th>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>KEY SKILLS</th>
          <th style='padding:12px 10px;text-align:left;color:#64748b;font-weight:600;font-size:12px'>SKILL GAPS</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table></div>"""

def format_shortlist_html(shortlisted: list) -> str:
    """Format shortlisted candidates as premium cards."""
    if not shortlisted:
        return "<p style='color:#888;text-align:center;padding:40px'>No shortlisted candidates yet.</p>"

    cards = ""
    medals = ["🥇", "🥈", "🥉", "4️⃣", "5️⃣"]
    for i, c in enumerate(shortlisted):
        score = c['overall_match_pct']
        color = '#22c55e' if score >= 80 else '#f59e0b' if score >= 65 else '#3b82f6'
        matched_skills = ", ".join(c.get('matched_skills', [])[:5]) or "N/A"
        gaps = ", ".join(c.get('skill_gaps', [])[:3]) or "None identified"

        cards += f"""
        <div style='background:white;border:1px solid #e2e8f0;border-radius:12px;padding:20px;margin-bottom:14px;border-left:4px solid {color};box-shadow:0 2px 8px rgba(0,0,0,0.06)'>
          <div style='display:flex;align-items:center;justify-content:space-between;margin-bottom:12px'>
            <div>
              <span style='font-size:20px'>{medals[i] if i < len(medals) else '🏅'}</span>
              <span style='font-size:16px;font-weight:700;color:#1e293b;margin-left:8px'>{c.get('name','N/A')}</span>
              <span style='background:#f1f5f9;color:#64748b;font-size:11px;padding:2px 8px;border-radius:20px;margin-left:8px'>ID: {c.get('candidate_id','')}</span>
            </div>
            <div style='font-size:28px;font-weight:800;color:{color}'>{score}%</div>
          </div>
          <div style='display:grid;grid-template-columns:1fr 1fr 1fr;gap:10px;font-size:12px'>
            <div><span style='color:#94a3b8'>Current Role</span><br><strong style='color:#334155'>{c.get('current_role','N/A')}</strong></div>
            <div><span style='color:#94a3b8'>Experience</span><br><strong style='color:#334155'>{c.get('experience_years',0)} years · {c.get('seniority_level','N/A')}</strong></div>
            <div><span style='color:#94a3b8'>Email</span><br><strong style='color:#334155'>{c.get('email','N/A')}</strong></div>
          </div>
          <div style='margin-top:10px;font-size:12px'>
            <span style='color:#94a3b8'>✅ Matched Skills:</span> <span style='color:#16a34a'>{matched_skills}</span>
          </div>
          <div style='margin-top:4px;font-size:12px'>
            <span style='color:#94a3b8'>⚠️ Skill Gaps:</span> <span style='color:#b45309'>{gaps}</span>
          </div>
        </div>"""

    return f"<div style='font-family:\"DM Sans\",sans-serif'>{cards}</div>"

print("✅ UI formatters ready!")

In [ ]:
# ============================================================
# CELL 14: MAIN ORCHESTRATOR FUNCTIONS
# ============================================================

# Global state storage
_global_state = {
    "scored_candidates": [],
    "shortlisted_candidates": [],
    "parsed_jd": {},
    "parsed_candidates": []
}

def run_full_pipeline(resume_files, jd_text, job_title, exp_level, shortlist_n, interview_type, comm_tone, comm_type):
    """Full pipeline orchestrator — runs LangGraph workflow."""
    global _global_state

    if not resume_files:
        return (
            "<p style='color:#ef4444;text-align:center;padding:30px'>⚠️ Please upload at least one resume.</p>",
            "<p>No candidates to display.</p>", "Waiting for input...",
            "Waiting for input...", "Waiting for input...", "Waiting for input..."
        )

    if not jd_text.strip():
        return (
            "<p style='color:#ef4444;text-align:center;padding:30px'>⚠️ Please enter a job description.</p>",
            "<p>No candidates to display.</p>", "Waiting for input...",
            "Waiting for input...", "Waiting for input...", "Waiting for input..."
        )

    # Extract resume texts
    resume_texts = []
    resume_filenames = []
    for f in resume_files:
        text = extract_resume_text(f.name)
        resume_texts.append(text)
        resume_filenames.append(Path(f.name).name)

    # Build initial state
    initial_state = RecruitmentState(
        resume_texts=resume_texts,
        resume_filenames=resume_filenames,
        job_description=jd_text,
        job_title=job_title,
        interview_type=interview_type.lower(),
        shortlist_count=int(shortlist_n),
        comm_type=comm_type.lower().replace(" ", "_"),
        comm_tone=comm_tone.lower(),
        parsed_candidates=[],
        parsed_jd={},
        scored_candidates=[],
        shortlisted_candidates=[],
        interview_questions="",
        interview_schedule="",
        hr_communication="",
        analytics_report="",
        status="Starting...",
        errors=[]
    )

    # Run LangGraph
    result = recruitment_graph.invoke(initial_state)

    # Store globally
    _global_state["scored_candidates"] = result["scored_candidates"]
    _global_state["shortlisted_candidates"] = result["shortlisted_candidates"]
    _global_state["parsed_jd"] = result["parsed_jd"]
    _global_state["parsed_candidates"] = result["parsed_candidates"]

    # Format outputs
    scorecard_html = format_scorecard_html(result["scored_candidates"])
    shortlist_html = format_shortlist_html(result["shortlisted_candidates"])
    interview_q = result.get("interview_questions", "No questions generated.")
    schedule = result.get("interview_schedule", "No schedule generated.")
    hr_comms = result.get("hr_communication", "No communication generated.")
    analytics = result.get("analytics_report", "No analytics generated.")

    return scorecard_html, shortlist_html, interview_q, schedule, hr_comms, analytics


def regenerate_questions(interview_type_sel):
    """Regenerate questions for currently shortlisted top candidate."""
    global _global_state
    if not _global_state["shortlisted_candidates"]:
        return "Please run the full pipeline first (Resume Screening tab)."
    top = _global_state["shortlisted_candidates"][0]
    return generate_interview_questions(top, _global_state["parsed_jd"], interview_type_sel.lower())


def regenerate_communication(comm_type_sel, comm_tone_sel, candidate_idx):
    """Regenerate HR communication for selected candidate."""
    global _global_state
    candidates = _global_state["shortlisted_candidates"]
    if not candidates:
        return "Please run the full pipeline first."
    idx = min(int(candidate_idx) - 1, len(candidates) - 1)
    idx = max(0, idx)
    candidate = candidates[idx]
    comm_map = {
        "Interview Invite": "shortlist_invite",
        "Rejection Email": "rejection",
        "Offer Letter": "offer_letter",
        "Onboarding Welcome": "onboarding_welcome",
        "Follow-up Email": "follow_up",
        "Assessment Task": "assessment_task",
        "Reference Check": "reference_check"
    }
    return generate_hr_communication(
        comm_map.get(comm_type_sel, "shortlist_invite"),
        candidate, _global_state["parsed_jd"], comm_tone_sel.lower()
    )


def run_workforce_planning(role_input, industry_input, team_size_input, growth_input):
    """Run workforce planning analysis."""
    return generate_workforce_plan(
        role_input, industry_input, int(team_size_input) if str(team_size_input).isdigit() else 10, growth_input
    )


def generate_jd_from_inputs(role_title_input, industry_inp, exp_level_inp, skills_inp):
    """Generate a new JD."""
    return generate_job_description_from_scratch(role_title_input, industry_inp, exp_level_inp, skills_inp)


print("✅ Orchestrator functions ready!")

In [ ]:
# ============================================================
# CELL 15: GRADIO UI — TALENTAI ENTERPRISE DASHBOARD
# ============================================================

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&family=Syne:wght@600;700;800&display=swap');

:root {
    --primary: #3b82f6;
    --primary-light: #eff6ff;
    --accent: #6366f1;
    --success: #22c55e;
    --warning: #f59e0b;
    --danger: #ef4444;
    --surface: #ffffff;
    --surface-2: #f8fafc;
    --border: #e2e8f0;
    --text: #1e293b;
    --text-muted: #64748b;
    --radius: 10px;
    --shadow: 0 2px 12px rgba(0,0,0,0.08);
}

body, .gradio-container {
    font-family: 'DM Sans', sans-serif !important;
    background: #f0f4ff !important;
    color: var(--text) !important;
}

/* Header marquee */
.talent-header {
    background: linear-gradient(135deg, #f0f7ff 0%, #e8f4fd 50%, #f0f0ff 100%);
    border-bottom: 2px solid #dbeafe;
    padding: 0;
    overflow: hidden;
    border-radius: 12px;
    margin-bottom: 16px;
    box-shadow: 0 2px 16px rgba(59,130,246,0.12);
}

.talent-logo-row {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 18px 28px;
    border-bottom: 1px solid #dbeafe;
}

.talent-logo {
    font-family: 'Syne', sans-serif;
    font-size: 26px;
    font-weight: 800;
    background: linear-gradient(135deg, #3b82f6, #6366f1);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    letter-spacing: -0.5px;
}

.talent-badge {
    background: linear-gradient(135deg, #3b82f6, #6366f1);
    color: white;
    font-size: 11px;
    font-weight: 600;
    padding: 4px 12px;
    border-radius: 20px;
    letter-spacing: 0.5px;
}

.marquee-container {
    overflow: hidden;
    padding: 10px 0;
    background: linear-gradient(90deg, #f0f7ff, #e8f4fd, #f0f0ff);
}

.marquee-track {
    display: flex;
    animation: marquee 28s linear infinite;
    white-space: nowrap;
    gap: 60px;
    align-items: center;
}

.marquee-item {
    font-size: 12px;
    font-weight: 600;
    color: #3b82f6;
    opacity: 0.8;
    letter-spacing: 0.3px;
}

.marquee-dot {
    color: #6366f1;
    font-size: 16px;
}

@keyframes marquee {
    0% { transform: translateX(0); }
    100% { transform: translateX(-50%); }
}

/* Main container */
.main-container {
    max-width: 1300px;
    margin: 0 auto;
    padding: 0 16px;
}

/* Cards */
.gr-box, .gr-group {
    background: var(--surface) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius) !important;
    box-shadow: var(--shadow) !important;
}

/* Tabs */
.tab-nav button {
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    color: var(--text-muted) !important;
    border-radius: 8px 8px 0 0 !important;
    padding: 10px 18px !important;
    transition: all 0.2s !important;
}

.tab-nav button.selected {
    color: var(--primary) !important;
    border-bottom: 2px solid var(--primary) !important;
    background: var(--primary-light) !important;
}

/* Buttons */
button.primary {
    background: linear-gradient(135deg, #3b82f6, #6366f1) !important;
    border: none !important;
    border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    padding: 10px 20px !important;
    color: white !important;
    box-shadow: 0 4px 14px rgba(59,130,246,0.3) !important;
    transition: all 0.2s !important;
}

button.primary:hover {
    transform: translateY(-1px) !important;
    box-shadow: 0 6px 20px rgba(59,130,246,0.4) !important;
}

button.secondary {
    background: white !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 8px !important;
    font-family: 'DM Sans', sans-serif !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    color: var(--text) !important;
}

/* Labels */
label.gr-label, .gr-label {
    font-family: 'DM Sans', sans-serif !important;
    font-size: 12px !important;
    font-weight: 600 !important;
    color: var(--text-muted) !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}

/* Input fields */
input[type=text], textarea, select {
    font-family: 'DM Sans', sans-serif !important;
    border-radius: 8px !important;
    border: 1.5px solid var(--border) !important;
    font-size: 13px !important;
    transition: border-color 0.2s !important;
}

input:focus, textarea:focus {
    border-color: var(--primary) !important;
    outline: none !important;
    box-shadow: 0 0 0 3px rgba(59,130,246,0.1) !important;
}

/* Section headers */
.section-header {
    font-family: 'Syne', sans-serif;
    font-size: 15px;
    font-weight: 700;
    color: var(--text);
    margin-bottom: 12px;
    padding-bottom: 8px;
    border-bottom: 2px solid var(--primary-light);
    display: flex;
    align-items: center;
    gap: 8px;
}

/* KPI Cards */
.kpi-row {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
    margin-bottom: 16px;
}

.kpi-card {
    background: white;
    border: 1px solid var(--border);
    border-radius: 10px;
    padding: 16px;
    text-align: center;
    box-shadow: var(--shadow);
}

.kpi-value {
    font-family: 'Syne', sans-serif;
    font-size: 28px;
    font-weight: 800;
    color: var(--primary);
}

.kpi-label {
    font-size: 11px;
    color: var(--text-muted);
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

/* Scrollable areas */
.scroll-area {
    max-height: 480px;
    overflow-y: auto;
    padding-right: 4px;
}

.scroll-area::-webkit-scrollbar {
    width: 4px;
}

.scroll-area::-webkit-scrollbar-track {
    background: #f1f5f9;
    border-radius: 4px;
}

.scroll-area::-webkit-scrollbar-thumb {
    background: #cbd5e1;
    border-radius: 4px;
}

/* Footer */
.talent-footer {
    text-align: center;
    padding: 16px;
    color: var(--text-muted);
    font-size: 12px;
    margin-top: 20px;
    border-top: 1px solid var(--border);
}
"""

HEADER_HTML = """
<div class='talent-header'>
  <div class='talent-logo-row'>
    <div class='talent-logo'>⚡ TalentAI</div>
    <div style='text-align:center'>
      <div style='font-family:"Syne",sans-serif;font-size:18px;font-weight:800;color:#1e293b'>Recruitment Intelligence Agent</div>
      <div style='font-size:12px;color:#64748b;margin-top:2px'>LangGraph · Groq LLaMA · AI-Powered Hiring Platform</div>
    </div>
    <div class='talent-badge'>by Aigenthix</div>
  </div>
  <div class='marquee-container'>
    <div class='marquee-track'>
      <span class='marquee-item'>📄 Resume Parsing & Extraction</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>🎯 AI-Powered Candidate Matching</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>⭐ Intelligent Shortlisting</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📅 Smart Interview Scheduling</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>🧠 Tailored Interview Questions</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📊 Hiring Analytics Dashboard</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>✉️ Automated HR Communications</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📈 Workforce Planning Intelligence</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📄 Resume Parsing & Extraction</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>🎯 AI-Powered Candidate Matching</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>⭐ Intelligent Shortlisting</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📅 Smart Interview Scheduling</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>🧠 Tailored Interview Questions</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📊 Hiring Analytics Dashboard</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>✉️ Automated HR Communications</span><span class='marquee-dot'>·</span>
      <span class='marquee-item'>📈 Workforce Planning Intelligence</span><span class='marquee-dot'>·</span>
    </div>
  </div>
</div>
"""

SAMPLE_JD = """Senior Data Scientist - Machine Learning Platform

We are seeking a highly skilled Senior Data Scientist to join our ML Platform team. 
You will lead the development of ML models, build data pipelines, and collaborate with 
engineering teams to deploy production ML systems at scale.

Requirements:
- 5+ years experience in data science or ML engineering
- Strong proficiency in Python, SQL, and ML frameworks (TensorFlow, PyTorch, scikit-learn)
- Experience with cloud platforms (AWS, GCP, or Azure)
- Knowledge of MLOps practices, CI/CD for ML, model monitoring
- Experience with big data tools: Spark, Kafka, Airflow
- Strong statistical knowledge and experimentation (A/B testing)
- Experience with NLP, Computer Vision, or Recommendation Systems preferred
- Excellent communication and team collaboration skills
- Masters or PhD in Computer Science, Statistics, or related field preferred

Responsibilities:
- Design and implement ML models for business impact
- Build and maintain ML training and inference pipelines
- Mentor junior data scientists
- Collaborate with product and engineering teams
- Present findings to executive stakeholders

Location: Remote-friendly | Salary: Competitive"""

# ─── Build Gradio App ─────────────────────────────────────────

with gr.Blocks(css=CUSTOM_CSS, title="TalentAI | Recruitment Intelligence Agent", theme=gr.themes.Soft()) as app:

    gr.HTML(HEADER_HTML)

    with gr.Tabs() as tabs:

        # ──────────────────────────────────────────────────────
        # TAB 1: RESUME SCREENING
        # ──────────────────────────────────────────────────────
        with gr.TabItem("📄 Resume Screening"):
            gr.HTML("<div class='section-header'>📄 Resume Screening & Job Description</div>")

            with gr.Row():
                with gr.Column(scale=1):
                    gr.HTML("<b style='font-size:13px;color:#334155'>Upload Resumes</b>")
                    resume_upload = gr.File(
                        label="Upload Resumes (PDF, DOCX, TXT)",
                        file_count="multiple",
                        file_types=[".pdf", ".docx", ".doc", ".txt"]
                    )

                    gr.HTML("<b style='font-size:13px;color:#334155;margin-top:16px;display:block'>Configuration</b>")
                    job_title_dd = gr.Dropdown(
                        label="Job Role",
                        choices=["Data Scientist", "ML Engineer", "Software Engineer", "Product Manager",
                                 "DevOps Engineer", "Frontend Developer", "Backend Developer", "Full Stack Developer",
                                 "Data Engineer", "AI Engineer", "Cloud Architect", "Cybersecurity Analyst",
                                 "HR Manager", "Business Analyst", "UX Designer", "Other"],
                        value="Data Scientist"
                    )
                    exp_level_dd = gr.Dropdown(
                        label="Experience Level",
                        choices=["Intern", "Junior (0-2 yrs)", "Mid (2-5 yrs)", "Senior (5-8 yrs)",
                                 "Lead (8-12 yrs)", "Manager (10+ yrs)", "Director (15+ yrs)"],
                        value="Senior (5-8 yrs)"
                    )
                    shortlist_n = gr.Slider(label="Shortlist Top N Candidates", minimum=1, maximum=15, step=1, value=5)
                    interview_type_dd = gr.Dropdown(
                        label="Interview Type",
                        choices=["Technical", "HR", "Managerial", "Mixed"],
                        value="Mixed"
                    )
                    comm_tone_dd = gr.Dropdown(
                        label="Communication Tone",
                        choices=["Professional", "Friendly", "Formal", "Warm", "Direct"],
                        value="Professional"
                    )
                    comm_type_dd = gr.Dropdown(
                        label="Communication Type (for HR tab)",
                        choices=["Interview Invite", "Rejection Email", "Offer Letter",
                                 "Onboarding Welcome", "Follow-up Email", "Assessment Task", "Reference Check"],
                        value="Interview Invite"
                    )

                    with gr.Row():
                        analyze_btn = gr.Button("🔍 Analyze Resumes", variant="primary")
                        clear_btn = gr.Button("🗑 Clear", variant="secondary")

                with gr.Column(scale=2):
                    jd_input = gr.Textbox(
                        label="Job Description",
                        placeholder="Paste your complete job description here...",
                        lines=18,
                        value=SAMPLE_JD
                    )

            gr.HTML("<div class='section-header' style='margin-top:20px'>📊 Candidate Scorecard</div>")
            scorecard_output = gr.HTML("<p style='color:#94a3b8;text-align:center;padding:40px;font-size:14px'>Upload resumes and click Analyze to see the scorecard...</p>")

        # ──────────────────────────────────────────────────────
        # TAB 2: CANDIDATE MATCHING
        # ──────────────────────────────────────────────────────
        with gr.TabItem("⭐ Candidate Matching"):
            gr.HTML("<div class='section-header'>⭐ Shortlisted Candidates — Top Matches</div>")
            gr.HTML("<p style='color:#64748b;font-size:13px;margin-bottom:16px'>Automatically shortlisted based on match score, skill alignment, and experience fit.</p>")
            shortlist_output = gr.HTML("<p style='color:#94a3b8;text-align:center;padding:40px;font-size:14px'>Run resume screening to see shortlisted candidates...</p>")

        # ──────────────────────────────────────────────────────
        # TAB 3: INTERVIEW PREPARATION
        # ──────────────────────────────────────────────────────
        with gr.TabItem("🧠 Interview Prep"):
            gr.HTML("<div class='section-header'>🧠 Interview Question Generator</div>")
            with gr.Row():
                interview_type_regen = gr.Dropdown(
                    label="Interview Type",
                    choices=["Technical", "HR", "Managerial", "Mixed"],
                    value="Technical",
                    scale=2
                )
                regen_q_btn = gr.Button("🔄 Regenerate Questions", variant="primary", scale=1)

            interview_q_output = gr.Textbox(
                label="Interview Questions & Evaluation Guide",
                lines=28,
                placeholder="Run resume screening first, then generate questions here...",
                interactive=False
            )

        # ──────────────────────────────────────────────────────
        # TAB 4: INTERVIEW SCHEDULING
        # ──────────────────────────────────────────────────────
        with gr.TabItem("📅 Scheduling"):
            gr.HTML("<div class='section-header'>📅 Interview Schedule Builder</div>")
            gr.HTML("<p style='color:#64748b;font-size:13px;margin-bottom:16px'>Multi-round interview schedule with panel composition and logistics.</p>")
            schedule_output = gr.Textbox(
                label="Generated Interview Schedule",
                lines=28,
                placeholder="Schedule will appear here after running resume screening...",
                interactive=False
            )

        # ──────────────────────────────────────────────────────
        # TAB 5: HR COMMUNICATION
        # ──────────────────────────────────────────────────────
        with gr.TabItem("✉️ HR Communication"):
            gr.HTML("<div class='section-header'>✉️ HR Communication Generator</div>")
            with gr.Row():
                comm_type_regen = gr.Dropdown(
                    label="Communication Type",
                    choices=["Interview Invite", "Rejection Email", "Offer Letter",
                             "Onboarding Welcome", "Follow-up Email", "Assessment Task", "Reference Check"],
                    value="Interview Invite"
                )
                comm_tone_regen = gr.Dropdown(
                    label="Tone",
                    choices=["Professional", "Friendly", "Formal", "Warm", "Direct"],
                    value="Professional"
                )
                candidate_idx_sel = gr.Slider(
                    label="Candidate # (from shortlist)",
                    minimum=1, maximum=10, step=1, value=1
                )
                regen_comm_btn = gr.Button("✉️ Generate Communication", variant="primary")

            hr_comm_output = gr.Textbox(
                label="Generated HR Communication",
                lines=28,
                placeholder="HR communication will appear here...",
                interactive=False
            )

            gr.HTML("<div class='section-header' style='margin-top:20px'>📝 JD Generator</div>")
            with gr.Row():
                jd_role_inp = gr.Textbox(label="Role Title", placeholder="e.g. ML Engineer", scale=2)
                jd_industry_inp = gr.Textbox(label="Industry", placeholder="e.g. Fintech", scale=1)
                jd_level_inp = gr.Dropdown(
                    label="Level",
                    choices=["Junior", "Mid", "Senior", "Lead", "Manager"],
                    value="Senior", scale=1
                )
            jd_skills_inp = gr.Textbox(label="Key Skills (comma separated)", placeholder="Python, ML, AWS, SQL")
            gen_jd_btn = gr.Button("📝 Generate Job Description", variant="primary")
            jd_gen_output = gr.Textbox(label="Generated Job Description", lines=16, interactive=False)

        # ──────────────────────────────────────────────────────
        # TAB 6: HIRING ANALYTICS
        # ──────────────────────────────────────────────────────
        with gr.TabItem("📊 Analytics"):
            gr.HTML("<div class='section-header'>📊 Hiring Analytics & Insights</div>")
            analytics_output = gr.Textbox(
                label="AI-Powered Hiring Analytics Report",
                lines=32,
                placeholder="Analytics report will appear here after running resume screening...",
                interactive=False
            )

        # ──────────────────────────────────────────────────────
        # TAB 7: WORKFORCE PLANNING
        # ──────────────────────────────────────────────────────
        with gr.TabItem("📈 Workforce Planning"):
            gr.HTML("<div class='section-header'>📈 Strategic Workforce Planning</div>")
            gr.HTML("<p style='color:#64748b;font-size:13px;margin-bottom:16px'>AI-powered talent strategy and hiring roadmap generation.</p>")

            with gr.Row():
                wf_role = gr.Textbox(label="Role/Function", placeholder="e.g. Data Science Team", scale=2)
                wf_industry = gr.Dropdown(
                    label="Industry",
                    choices=["Technology", "Finance", "Healthcare", "E-Commerce", "Manufacturing",
                             "Consulting", "Automotive", "Telecom", "Retail", "Education"],
                    value="Technology", scale=1
                )
            with gr.Row():
                wf_team_size = gr.Slider(label="Current Team Size", minimum=1, maximum=200, step=1, value=10)
                wf_growth = gr.Dropdown(
                    label="Growth Target",
                    choices=["Hire 1-2 people", "Hire 3-5 people", "Scale team 2x",
                             "Scale team 3x", "Reduce headcount", "Restructure team"],
                    value="Hire 3-5 people"
                )
            wf_plan_btn = gr.Button("📈 Generate Workforce Plan", variant="primary")
            wf_plan_output = gr.Textbox(label="Strategic Workforce Plan", lines=28, interactive=False)

    gr.HTML("""
    <div class='talent-footer'>
      ⚡ TalentAI — Recruitment Intelligence Agent &nbsp;|&nbsp;
      Built by <strong>Aigenthix</strong> &nbsp;|&nbsp;
      Powered by LangGraph · Groq LLaMA 3.1 · Gradio
    </div>
    """)

    # ─── Wire Events ─────────────────────────────────────────

    analyze_btn.click(
        fn=run_full_pipeline,
        inputs=[resume_upload, jd_input, job_title_dd, exp_level_dd,
                shortlist_n, interview_type_dd, comm_tone_dd, comm_type_dd],
        outputs=[scorecard_output, shortlist_output, interview_q_output,
                 schedule_output, hr_comm_output, analytics_output]
    )

    clear_btn.click(
        fn=lambda: [
            "<p style='color:#94a3b8;text-align:center;padding:40px'>Cleared. Ready for new input.</p>",
            "<p style='color:#94a3b8;text-align:center;padding:40px'>Cleared.</p>",
            "", "", "", ""
        ],
        outputs=[scorecard_output, shortlist_output, interview_q_output,
                 schedule_output, hr_comm_output, analytics_output]
    )

    regen_q_btn.click(
        fn=regenerate_questions,
        inputs=[interview_type_regen],
        outputs=[interview_q_output]
    )

    regen_comm_btn.click(
        fn=regenerate_communication,
        inputs=[comm_type_regen, comm_tone_regen, candidate_idx_sel],
        outputs=[hr_comm_output]
    )

    gen_jd_btn.click(
        fn=generate_jd_from_inputs,
        inputs=[jd_role_inp, jd_industry_inp, jd_level_inp, jd_skills_inp],
        outputs=[jd_gen_output]
    )

    wf_plan_btn.click(
        fn=run_workforce_planning,
        inputs=[wf_role, wf_industry, wf_team_size, wf_growth],
        outputs=[wf_plan_output]
    )

print("✅ Gradio UI built successfully!")

In [ ]:
# ============================================================
# CELL 16: LAUNCH THE APP
# ============================================================

print("🚀 Launching TalentAI — Recruitment Intelligence Agent...")
print("📡 By Aigenthix | LangGraph + Groq + Gradio")
print("=" * 60)

app.launch(
    share=True,       # Creates a public Gradio link
    debug=False,
    show_error=True,
    quiet=False
)

# ─── If share=True doesn't work, try:
# app.launch(server_name="0.0.0.0", server_port=7860, share=False)

In [ ]:
# ============================================================
# CELL 17: SAMPLE DEMO (Text-Only, No File Upload)
# ============================================================
# Run this cell to test the AI pipeline WITHOUT uploading files
# Uses sample resume text directly.

SAMPLE_RESUME_1 = """
Priya Sharma | priya.sharma@email.com | +91-9876543210 | Bangalore, India
LinkedIn: linkedin.com/in/priyasharma

PROFESSIONAL SUMMARY
Senior Data Scientist with 7 years of experience in machine learning, deep learning,
and production ML systems. Led 15+ ML projects across e-commerce and fintech domains.

EXPERIENCE
Senior Data Scientist | Flipkart | 2020-Present (4 years)
- Built recommendation engine serving 50M+ users using collaborative filtering + deep learning
- Led NLP project for customer sentiment analysis (BERT fine-tuning)
- Developed MLOps pipeline using MLflow, Kubeflow, and Airflow
- A/B tested 20+ ML models, improved CTR by 18%
- Tech: Python, PyTorch, TensorFlow, Spark, GCP, BigQuery, Docker, Kubernetes

Data Scientist | Paytm | 2018-2020 (2 years)
- Fraud detection model (XGBoost) — reduced fraud by 32%
- Built credit risk scoring pipeline
- Tech: Python, scikit-learn, SQL, AWS, Kafka

Junior Data Analyst | TCS | 2017-2018 (1 year)
- Data analysis and reporting, SQL, Tableau

EDUCATION
M.Tech in Computer Science | IIT Bombay | 2017 | Specialization: AI/ML
B.Tech in Computer Science | NIT Trichy | 2015

TECHNICAL SKILLS
Languages: Python, SQL, Scala, R
ML/DL: TensorFlow, PyTorch, scikit-learn, XGBoost, LightGBM, BERT, GPT
MLOps: MLflow, Kubeflow, Airflow, Docker, Kubernetes, CI/CD
Cloud: GCP (Vertex AI, BigQuery, GCS), AWS (SageMaker, S3, EMR)
Big Data: Apache Spark, Kafka, Hive, Hadoop
Tools: Git, JIRA, Confluence, Tableau

CERTIFICATIONS
- Google Professional ML Engineer
- AWS Certified ML Specialty
- Deep Learning Specialization (Coursera/deeplearning.ai)

ACHIEVEMENTS
- Best ML Project Award, Flipkart Data Science Summit 2022
- Published paper on recommendation systems at RecSys 2021
- Kaggle Expert (top 5% global ranking)
"""

SAMPLE_RESUME_2 = """
Rahul Verma | rahul.v@email.com | +91-8765432109 | Hyderabad, India

SUMMARY
Data Scientist with 3 years experience in NLP and computer vision. Good knowledge of
Python and machine learning fundamentals. Looking to grow into senior roles.

EXPERIENCE
Data Scientist | Infosys BPO | 2022-Present (2 years)
- Built text classification models using traditional ML (SVM, Naive Bayes)
- Developed image recognition POC using CNN
- Python, scikit-learn, OpenCV, MySQL, basic AWS

Data Analyst | Wipro | 2021-2022 (1 year)
- SQL reporting, Excel dashboards, Power BI
- Basic Python scripting for data cleaning

EDUCATION
B.Tech Computer Science | Osmania University | 2021

SKILLS
Python, scikit-learn, basic TensorFlow, SQL, Power BI, Excel
Limited experience with cloud (AWS basics)
No production ML deployment experience
"""

SAMPLE_RESUME_3 = """
Anika Singh | anika.singh@techmail.com | Mumbai, India | +91-7654321098

PROFILE
Staff ML Engineer / Lead Data Scientist with 10 years building ML infrastructure and
data-intensive products. Managed teams of 8-12 data scientists. Deep expertise in
production ML systems, real-time inference, and data platform engineering.

WORK HISTORY
Lead ML Engineer | Swiggy | 2019-Present (5 years)
- Architected real-time recommendation and ETA prediction systems
- Led team of 8 data scientists and ML engineers
- Tech stack: Python, PyTorch, TensorFlow, Flink, Kafka, Kubernetes, AWS SageMaker
- Reduced model inference latency by 60% through optimization

Senior Data Scientist | Amazon India | 2016-2019 (3 years)
- Demand forecasting models (LSTM, Prophet)
- Supply chain optimization
- Python, Spark, AWS, Redshift

Data Scientist | Mu Sigma | 2014-2016 (2 years)

EDUCATION
M.S. Statistics | IIT Delhi | 2014
B.Sc. Mathematics | Delhi University | 2012

SKILLS
Python (expert), PyTorch, TensorFlow, JAX
Spark, Kafka, Flink, Airflow, Kubeflow
AWS (SageMaker, EMR, Lambda, S3), GCP
Kubernetes, Docker, MLflow, Feature Stores
Deep Learning, NLP, Computer Vision, Causal ML

CERTIFICATIONS
AWS ML Specialty, GCP Professional Data Engineer

PUBLICATIONS
3 papers at NeurIPS, ICML workshops on production ML
"""

print("🧪 Running Demo Pipeline with 3 sample candidates...")
print("=" * 60)

# Parse JD
parsed_jd = parse_job_description(SAMPLE_JD)
print(f"✅ JD Parsed: {parsed_jd.get('job_title')}")
print(f"   Required Skills: {parsed_jd.get('required_technical_skills', [])[:5]}")
print(f"   Required Experience: {parsed_jd.get('required_experience_years')} years")
print()

# Parse Candidates
resumes = [SAMPLE_RESUME_1, SAMPLE_RESUME_2, SAMPLE_RESUME_3]
names = ["Priya Sharma", "Rahul Verma", "Anika Singh"]

parsed_candidates = []
for resume, name in zip(resumes, names):
    parsed = parse_candidate_features(resume, name)
    parsed_candidates.append(parsed)
    print(f"✅ Parsed: {parsed.get('name')} | Skills: {parsed.get('technical_skills', [])[:4]}")

# Score Candidates
print("\n📊 Scoring candidates...")
scored = [score_candidate(c, parsed_jd) for c in parsed_candidates]
scored.sort(key=lambda x: x['overall_match_pct'], reverse=True)

print("\n🏆 CANDIDATE RANKINGS:")
print("-" * 55)
for i, c in enumerate(scored, 1):
    print(f"#{i} {c['name']:<20} | Match: {c['overall_match_pct']:>3}% | {c['rating']}")
    print(f"   Exp: {c['experience_years']}y | Level: {c['seniority_level']}")
    if c['skill_gaps']:
        print(f"   Gaps: {', '.join(c['skill_gaps'][:3])}")
    print()

# Shortlist
shortlisted = shortlist_candidates(scored, n=3)
print(f"✅ Shortlisted {len(shortlisted)} candidates")

# Generate a sample interview question snippet
print("\n🧠 Generating interview questions for top candidate...")
q_output = generate_interview_questions(shortlisted[0], parsed_jd, "technical")
print(q_output[:600] + "...\n")

print("\n✅ Demo complete! Launch the Gradio UI (Cell 16) for the full interactive experience.")